> **Meridian AgentOps workshop · notebook 01 of 6 (Module 3).** The reusable code lives in the
> repo's `app/` package (agent, tools, MCP service desk, evaluators, config) — these notebooks
> import it, so a fresh session only needs the bootstrap cells below instead of re-running
> earlier modules. **Prerequisite:** notebook 00 has run once against your Langfuse project (prompts + agent exist).


## Module 3 · Prompt versioning 


- Why it matters: the prompt **is** the behaviour of your agent, but it shouldn't live inside your code.
- In Langfuse, a **prompt** has immutable numbered **versions**; movable **labels** (`production`, `latest`,
anything custom) are pointers to versions. Your app fetches *by label* 

In [ ]:
# ── 0.1 Get the code + the pinned stack (fresh Colab VM: clone first) ──
import os
if not os.path.isdir("../app"):                    # fresh Colab VM → clone the repo
    !git clone https://github.com/kartik-nighania/data-hack-summit-2026.git _workshop_repo
    %cd _workshop_repo/workshop
%pip install -q -r ../requirements.txt
print("✅ stack ready — if pip just upgraded packages, do Run ▸ Restart session once and rerun from the top.")


In [ ]:
import os, sys

sys.path.insert(0, os.path.abspath(".."))        # make the repo's app/ package importable

# Jupyter kernels already run an event loop; this lets libraries that call
# asyncio.run()/run_until_complete work inside notebook cells.
import nest_asyncio
nest_asyncio.apply()

print("✅ environment prepared |", sys.version.split()[0])


In [ ]:
# ── 0.3 Load API keys from .env (copy .env.example → .env next to requirements.txt) ─
from app.config import load_keys
load_keys()


In [ ]:
# ── 0.4 Constants (config.yaml) + the Langfuse client (PII masking hook registered) ─
from app.config import get_lf
lf = get_lf()
if not lf.auth_check():
    raise SystemExit("❌ Langfuse authentication FAILED. Check keys + LANGFUSE_HOST region "
                     "(EU: https://cloud.langfuse.com / US: https://us.cloud.langfuse.com).")
LANGFUSE_HOST = os.environ["LANGFUSE_HOST"]
import importlib.metadata as _md
print("✅ Langfuse authenticated:", LANGFUSE_HOST)
for p in ["langfuse", "langchain", "langgraph", "deepeval", "openai", "fastmcp"]:
    print(f"   {p}=={_md.version(p)}")


In [ ]:
# Session imports: the deployed agent from app/
from app.agent import run_agent


In [ ]:
# ── 3.1 Fetching: by label, by version, and what's inside a prompt object ────
p_prod   = lf.get_prompt("final-response", type="chat", cache_ttl_seconds=0)              # default: production label
p_latest = lf.get_prompt("final-response", type="chat", label="latest", cache_ttl_seconds=0)
p_v1     = lf.get_prompt("final-response", type="chat", version=1, cache_ttl_seconds=0)

print(f"production → v{p_prod.version} | latest → v{p_latest.version} | explicit v1 → v{p_v1.version}")
print("labels on production version:", p_prod.labels)
print("config travels with the version:", p_prod.config)
print("\nfirst 140 chars of the system message:\n", p_prod.prompt[0]["content"][:140], "…")
# NOTE cache_ttl_seconds=0 → every fetch hits the API so your UI edits apply instantly.
# In production you'd keep the default 60s cache (stale-while-revalidate, never blocks).

In [ ]:
# ── 3.2 Ship a new version to STAGING (not to customers yet) ─────────────────
current = lf.get_prompt("final-response", type="chat", label="latest", cache_ttl_seconds=0)

def as_editable(prompt_obj):
    """Fetched prompt → the message form create_prompt accepts (strip read-only keys)."""
    out = []
    for m in prompt_obj.prompt:
        if m.get("type") == "placeholder":
            out.append({"type": "placeholder", "name": m["name"]})
        else:
            out.append({"role": m["role"], "content": m["content"]})
    return out

staged_messages = as_editable(current)
staged_messages[0]["content"] += "\nSign off exactly as: 'Team Meridian'."
if "Team Meridian" not in current.prompt[0]["content"]:
    lf.create_prompt(name="final-response", type="chat", prompt=staged_messages,
                     labels=["staging"], config=current.config)   # new version, staging label only

staged = lf.get_prompt("final-response", type="chat", label="staging", cache_ttl_seconds=0)
prod   = lf.get_prompt("final-response", type="chat", cache_ttl_seconds=0)
print(f"staging → v{staged.version} (has the new sign-off) | production → v{prod.version} (unchanged)")
print("Customers are still served production. Open Langfuse ▸ Prompts ▸ final-response to see both versions.")

In [ ]:
# ── 3.3 Promote to production … then ROLL BACK with one label move ───────────
staged_v, original_v = staged.version, prod.version

lf.update_prompt(name="final-response", version=staged_v, new_labels=["production"])
now = lf.get_prompt("final-response", type="chat", cache_ttl_seconds=0)
print(f"PROMOTED: production now serves v{now.version} — the very next customer request uses it. No redeploy.")

r = await run_agent("What's my outstanding balance?", "CUST-1002", session_id="sess-m3-promote", tags=["module-3"])
print("\nReply now ends with:", r["answer"].strip().splitlines()[-1])

lf.update_prompt(name="final-response", version=original_v, new_labels=["production"])
back = lf.get_prompt("final-response", type="chat", cache_ttl_seconds=0)
print(f"\nROLLED BACK: production serves v{back.version} again. That's the whole rollback procedure —")
print("remember it: it is exactly what we'll do during the Module 9 incident.")
lf.flush()

In [ ]:
# ── 3.4 Resilience: fallbacks for when Langfuse is unreachable at cold start ─
p = lf.get_prompt(
    "final-response", type="chat", cache_ttl_seconds=0,
    fallback=[{"role": "system", "content": "You are Meridian's support desk. Answer from the conversation."},
              {"type": "placeholder", "name": "conversation"}],
)
print("is_fallback:", p.is_fallback, "→ False because the API is reachable; the fallback only kicks in when")
print("there is no cached copy AND the fetch fails. Caveat: fallback prompts create NO prompt↔trace link.")

### ✅ CHECKPOINT — prompts as deployments
- In **Langfuse ▸ Prompts ▸ final-response** you can see the version list with labels moving between versions
(the promote + rollback you just did).
- **Final Response ▸ Metrics**: shows usage between both versions